# 04_transfer_finetune_B.ipynb
Target-domain adaptation on B only.


In [ ]:
from pathlib import Path
import json, numpy as np, tensorflow as tf
BUILD_DIR=Path('dataset_build_hybrid'); SRC_DIR=Path('source_train_runs'); OUT_DIR=Path('transfer_B_runs'); OUT_DIR.mkdir(exist_ok=True)

def load_npz(name):
    p=BUILD_DIR/f'{name}.npz'
    if not p.exists(): return None
    o=np.load(p,allow_pickle=True); return {k:o[k] for k in o.files}
B_finetune=load_npz('B_finetune'); B_val=load_npz('B_val'); target_eval_unseen_locations=load_npz('target_eval_unseen_locations')
assert B_finetune is not None and target_eval_unseen_locations is not None
model=tf.keras.models.load_model(SRC_DIR/'hybrid_source_A_best.keras',compile=False)


In [ ]:
labels=sorted(set(B_finetune['anchor_label'].astype(str).tolist() + ([] if B_val is None else B_val['anchor_label'].astype(str).tolist())))
label_to_id={k:i for i,k in enumerate(labels)}
def make_arrays(obj):
    Xamp=[];Xpha=[];y=[]
    for i in range(len(obj['amp_path'])):
        amp=np.load(str(obj['amp_path'][i])).astype('float32'); pha=np.load(str(obj['pha_path'][i])).astype('float32') if Path(str(obj['pha_path'][i])).exists() else np.zeros_like(amp)
        if amp.ndim==2: amp=amp[...,None]
        if pha.ndim==2: pha=pha[...,None]
        Xamp.append(amp);Xpha.append(pha);y.append(label_to_id[str(obj['anchor_label'][i])])
    return np.stack(Xamp),np.stack(Xpha),tf.keras.utils.to_categorical(y,num_classes=len(label_to_id))
Xamp_ft,Xpha_ft,y_ft=make_arrays(B_finetune)


In [ ]:
for lyr in model.layers[:-4]: lyr.trainable=False
model.compile(optimizer=tf.keras.optimizers.Adam(5e-5),loss='categorical_crossentropy',metrics=['accuracy'])
# model.fit({'amp_in':Xamp_ft,'pha_in':Xpha_ft}, y_ft, ...)
for lyr in model.layers[-8:]: lyr.trainable=True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),loss='categorical_crossentropy',metrics=['accuracy'])
model.save(OUT_DIR/'hybrid_transfer_B_best.keras'); model.save(OUT_DIR/'hybrid_transfer_B_final.keras')
(OUT_DIR/'transfer_B_history.json').write_text(json.dumps({'phase1':'placeholder','phase2':'placeholder'},indent=2))
summary={'adaptation_domain':'B','used_split':'B_finetune','optional_val':'B_val','excluded_from_training':'target_eval_unseen_locations','single_receiver_only':True}
(OUT_DIR/'transfer_B_summary.json').write_text(json.dumps(summary,indent=2)); print(summary)
